# Buildings, people, and urban context

**Atlas source reading copy · browser-local**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Urbanity/blob/a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f/book/notebooks/urban-context.ipynb)



Standard-library synthetic network, context, imagery-label and graph exercises; these executable cells do not import Urbanity or download OSM/provider data. Native Urbanity, compiled GIS, imagery and graph-ML examples in Markdown are continuations requiring their own environment and data permissions. Owner-authorized original material has no inferred public repository license.

**Provenance:** `book/notebooks/urban-context.ipynb` at `a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f`; original SHA-256 `47052720b732f7999573bdd6723f2520c28baadae21d233b195650e296ef4c2e`. Original owner-authored companion incorporated at repository owner request; Urbanity source repository has no public license grant. Package Urbanity and third-party data retain separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Buildings, people, and urban context

Street structure becomes more interpretable when combined with other data. Building footprints and morphology, population grids, and land-use attributes can add context to places and network elements. Here we use a tiny synthetic table to practise aggregation; it is not a real population or building dataset.

![Illustrative urban-analysis pipeline](../_static/urbanity-pipeline.svg)

[Launch this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Urbanity/lab/lab/index.html?path=notebooks/urban-context.ipynb) · [Urbanity API reference](https://urbanity.readthedocs.io/en/latest/api/)

## Try it: summarize toy blocks

The rows below are constructed example values. Real analysis needs source metadata, compatible spatial reference systems, appropriate spatial joins, and checks for missing or duplicated records.

In [1]:
buildings = [
    {"block": "North", "footprint_m2": 120, "height_m": 12, "people": 30},
    {"block": "North", "footprint_m2": 180, "height_m": 18, "people": 45},
    {"block": "South", "footprint_m2": 90, "height_m": 6, "people": 12},
    {"block": "South", "footprint_m2": 240, "height_m": 24, "people": 60},
]

summary = {}
for building in buildings:
    block = building["block"]
    bucket = summary.setdefault(block, {"buildings": 0, "footprint_m2": 0, "people": 0})
    bucket["buildings"] += 1
    bucket["footprint_m2"] += building["footprint_m2"]
    bucket["people"] += building["people"]

for block, values in sorted(summary.items()):
    values["people_per_building"] = values["people"] / values["buildings"]
    print(block, values)

North {'buildings': 2, 'footprint_m2': 300, 'people': 75, 'people_per_building': 37.5}
South {'buildings': 2, 'footprint_m2': 330, 'people': 72, 'people_per_building': 36.0}


Aggregation can hide within-area differences. Two blocks with the same total population can have different distributions across buildings, and counts are not the same as density. Density needs a defined area denominator and compatible units.

In [2]:
def footprint_ratio(building, plot_area_m2):
    """Simple footprint-to-plot-area ratio for this toy example."""
    if plot_area_m2 <= 0:
        raise ValueError("plot area must be positive")
    return building["footprint_m2"] / plot_area_m2

plot_area_m2 = {"North": 900, "South": 1100}
for block, values in sorted(summary.items()):
    ratio = values["footprint_m2"] / plot_area_m2[block]
    print(f"{block}: total-footprint ratio = {ratio:.2f}")

North: total-footprint ratio = 0.33
South: total-footprint ratio = 0.30


## Native Urbanity pathways

Urbanity documents building morphology helpers, population-data utilities (including Meta and GHS workflows), and options to enrich network or graph layers. Consult the [building API](https://urbanity.readthedocs.io/en/latest/api/building/) and [population API](https://urbanity.readthedocs.io/en/latest/api/population/) for exact data sources and method signatures.

The documented building-shape measures include complexity, squareness, shape index, compactness, rectangularity, fractal dimension, equivalent rectangular index, longest-axis length, shared-wall ratio, orientation, elongation, corners, convexity, and circular compactness. These are geometric descriptors, not direct measures of building quality or lived experience.

A documented graph-building option, for example, requests population context while constructing an UrbanGraph:

```python
urban_graph = city.get_urban_graph(
    bandwidth=100,
    minimum_area=30,
    network_type="driving",
    population_layer="meta",
)
```

Urbanity also documents area-level aggregate statistics (including an optional street-view workflow) and raster-to-vector population utilities. The browser notebook does not fetch or process geospatial data. Dataset years, coverage, spatial resolution, population definitions, and redistribution terms can differ; inspect metadata and cite each input dataset independently.

## Reflect

- Which denominator would answer your question: plot area, land area, floor area, or people?
- How would you report uncertainty or missing building attributes?
- What privacy risks might arise when reporting fine-grained population data?